In [ ]:
from pynq import Overlay, allocate, PL
import numpy as np
import random
import time
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

PL.reset()
overlay = Overlay('design_6.bit')

In [ ]:
print('IP blocks :', list(overlay.ip_dict.keys()))

In [ ]:
gameoflife_hard = overlay.gameoflife_compute_0

In [ ]:
def get_register_offset(overlay, ip, parameter):
    return overlay.ip_dict[ip]['registers'][parameter]['address_offset']

In [ ]:
dma_send = overlay.axi_dma_0.sendchannel
dma_recv = overlay.axi_dma_0.recvchannel

CONTROL_REGISTER = 0x0

WIDTH_REGISTER = get_register_offset(overlay, 'gameoflife_compute_0', 'grid_width')
HEIGHT_REGISTER = get_register_offset(overlay, 'gameoflife_compute_0', 'grid_height')

In [ ]:
width = 512
height = 512

buffer1 = allocate(shape=(width*height,), dtype=np.int32)
buffer2 = allocate(shape=(width*height,), dtype=np.int32)

for y in range(height):
    for x in range(width):
        n = random.random()
        if n > 0.2:
            buffer1[y*width + x] = 1
        else:
            buffer1[y*width + x] = 0

In [ ]:
to_plot = buffer1.reshape((width, height))#.astype(float)
imgplot = plt.imshow(to_plot)
plt.show()

In [ ]:
def gameoflife_fpga(buffer_in, buffer_out, width, height):

    gameoflife_hard.write(WIDTH_REGISTER, width)
    gameoflife_hard.write(HEIGHT_REGISTER, height)    
    gameoflife_hard.write(CONTROL_REGISTER, 0x01)

    dma_send.transfer(buffer_in)
    dma_recv.transfer(buffer_out)
    dma_send.wait()
    dma_recv.wait()

In [ ]:
%matplotlib inline

import time
import matplotlib.pyplot as plt
from IPython.display import clear_output

for i in range(100):

    start_time = time.time()
    gameoflife_fpga(buffer1, buffer2, width, height)
    #swap buffers
    temp = buffer1
    buffer1 = buffer2
    buffer2 = temp
    processing_time = time.time() - start_time
        
    clear_output(wait=True)
    plt.imshow(buffer1.reshape((width, height)))
    plt.title(f"Iteration {i+1} time {processing_time}")
    plt.show()